# BM25 vs Vector Retrieval Smoke Test

This notebook compares the existing vector search with the new LangChain BM25 keyword search for the same query.

Vector search uses OpenAI embeddings, so it needs `OPENAI_API_KEY`. BM25 runs locally over the Chroma documents.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from somewhere inside the project checkout.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display
import pandas as pd

from backend.rag.search.hybrid_document_search import search_source_documents
from backend.rag.search.keyword_search import keyword_search
from backend.rag.search.vector_search import load_collection, search as vector_search

In [2]:
QUERY = "Thomas Janssen"
TOP_K = 5

openai_client, collection = load_collection()

vector_results = vector_search(
    openai_client=openai_client,
    collection=collection,
    query=QUERY,
    top_k=TOP_K,
)
keyword_results = keyword_search(
    collection=collection,
    query=QUERY,
    top_k=TOP_K,
)

print(f"query={QUERY!r}")
print(f"vector results: {len(vector_results)}")
print(f"BM25 keyword results: {len(keyword_results)}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


query='Thomas Janssen'
vector results: 5
BM25 keyword results: 5


In [3]:
def summarize_results(rows):
    records = []
    for rank, row in enumerate(rows, start=1):
        metadata = row.get("metadata") or {}
        text = " ".join(str(row.get("text") or "").split())
        records.append(
            {
                "rank": rank,
                "chunk_id": row.get("chunk_id"),
                "document_id": metadata.get("document_id"),
                "document_type": metadata.get("document_type"),
                "page": metadata.get("page_number"),
                "vector_distance": row.get("distance"),
                "keyword_rank": row.get("keyword_rank"),
                "source_rerank_score": row.get("source_rerank_score"),
                "methods": ", ".join(row.get("retrieval_methods", [])),
                "text": text[:450],
            }
        )
    return pd.DataFrame.from_records(records)

display(Markdown("### Vector Search Results"))
display(summarize_results(vector_results))

display(Markdown("### BM25 Keyword Search Results"))
display(summarize_results(keyword_results))

### Vector Search Results

,rank,chunk_id,document_id,document_type,page,vector_distance,keyword_rank,source_rerank_score,methods,text
0,1,donation_money_006-p002-c0004,donation_money_006,monetary_donation,2,0.955523,None,None,,**The Donors:** ______________________________...
1,2,certificate_016-p001-c0001,certificate_016,family_composition_certificate,1,1.037809,None,None,,| Name | Date of Birth | Relationship | |-----...
2,3,certificate_016-p001-c0002,certificate_016,family_composition_certificate,1,1.122941,None,None,,--- **III. RELATIONSHIP STATEMENTS** 1. Hendri...
3,4,poa_011-p003-c0004,poa_011,power_of_attorney,3,1.193285,None,None,,"IN WITNESS WHEREOF, the parties have executed ..."
4,5,will_012-p002-c0002,will_012,notarial_will,2,1.201909,None,None,,the residue of my estate shall be distributed ...


### BM25 Keyword Search Results

,rank,chunk_id,document_id,document_type,page,vector_distance,keyword_rank,source_rerank_score,methods,text
0,1,certificate_016-p001-c0001,certificate_016,family_composition_certificate,1,None,1,None,,| Name | Date of Birth | Relationship | |-----...
1,2,donation_money_006-p002-c0004,donation_money_006,monetary_donation,2,None,2,None,,**The Donors:** ______________________________...
2,3,certificate_016-p001-c0002,certificate_016,family_composition_certificate,1,None,3,None,,--- **III. RELATIONSHIP STATEMENTS** 1. Hendri...
3,4,mortgage_014-p003-c0005,mortgage_014,mortgage_deed,3,None,4,None,,--- **REGISTRATION AND COSTS** 8. This mortgag...
4,5,sale_004-p003-c0005,sale_004,sale_deed,3,None,5,None,,"**SIGNATURES:** Bruges, 14 October 2020 **For ..."


In [4]:
# Optional: this is the document candidate set your RAG flow now sends onward
# before the existing document+graph reranking step.
hybrid_results = search_source_documents(
    openai_client=openai_client,
    collection=collection,
    query=QUERY,
    top_k=TOP_K,
)

display(Markdown("### Hybrid Document Results After Document Reranking"))
display(summarize_results(hybrid_results))

/workspace/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/.venv/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'libc10_cuda.so: cannot open shared object file: No such file or directory'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/workspace/.venv/lib/python3.11/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Plea

### Hybrid Document Results After Document Reranking

,rank,chunk_id,document_id,document_type,page,vector_distance,keyword_rank,source_rerank_score,methods,text
0,1,certificate_016-p001-c0002,certificate_016,family_composition_certificate,1,1.122606,3.0,6.696317,"vector, keyword",--- **III. RELATIONSHIP STATEMENTS** 1. Hendri...
1,2,certificate_016-p001-c0001,certificate_016,family_composition_certificate,1,1.037551,1.0,5.856457,"vector, keyword",| Name | Date of Birth | Relationship | |-----...
2,3,donation_money_006-p002-c0004,donation_money_006,monetary_donation,2,0.955311,2.0,5.822229,"vector, keyword",**The Donors:** ______________________________...
3,4,sale_004-p003-c0005,sale_004,sale_deed,3,NaN,5.0,5.154706,keyword,"**SIGNATURES:** Bruges, 14 October 2020 **For ..."
4,5,will_012-p002-c0002,will_012,notarial_will,2,1.201685,NaN,4.706500,vector,the residue of my estate shall be distributed ...
